## Data Migration: SQL to Postgres

In [3]:
import os
import pandas as pd
import pyodbc
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv


## 1. LOAD CREDENTIALS

In [4]:
load_dotenv()

True

In [5]:
sql_host = os.getenv("SQLSERVER_HOST")
sql_db = os.getenv("SQLSERVER_DB")
sql_user = os.getenv("SQLSERVER_USER")
sql_password = os.getenv("SQLSERVER_PASSWORD")


In [6]:
print(f"SQL SERVER HOST:{sql_host}") 
print(f"SQL DB:{sql_db}") 
print(f"SQL USER:{sql_user}") 
print(f"SQL PASSWORD:{sql_password}") 

SQL SERVER HOST:DESKTOP-P6OS4A2\SQLEXPRESS
SQL DB:AdventureWorks2022
SQL USER:loadingusr
SQL PASSWORD:Toluca2000


In [7]:
pg_host= os.getenv("PostgreSQL_HOST")
pg_port= os.getenv("PostgreSQL_PORT")
pg_db= os.getenv("PostgreSQL_DB")
pg_user= os.getenv("PostgreSQL_USER")
pg_password= os.getenv("PostgreSQL_PASSWORD")

In [8]:
print(f"POSTGRE HOST:{pg_host}")
print(f"POSTGRE PORT:{pg_port}")
print(f"POSTGRE DB:{pg_db}")
print(f"POSTGRE USER:{pg_user}")
print(f"POSTGRE PASSWORD:{pg_password}")

POSTGRE HOST:localhost
POSTGRE PORT:5432
POSTGRE DB:dvdrental
POSTGRE USER:loadingusr
POSTGRE PASSWORD:Toluca2000


## Connect to SQL Server

In [9]:
print("connect to SQL Server...")
print(f"    Server:{sql_host}")
print(f"    User:{sql_user}")
print(f"    Password:{sql_password}")
print(f"    DB:{sql_db}")

connect to SQL Server...
    Server:DESKTOP-P6OS4A2\SQLEXPRESS
    User:loadingusr
    Password:Toluca2000
    DB:AdventureWorks2022


In [10]:
print(pyodbc.drivers())

['SQL Server', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 17 for SQL Server']


In [ ]:
try:
    sql_conn_string =(
        f"USER={sql_user};"
        f"PASSWORD={sql_password};"
        f"SERVER={sql_host};"
        f"DATABASE={sql_db};"
        f"DRIVER={{SQL Server}};"
        f"Trusted_Conection=yes;"
    )
    sql_conn = pyodbc.connect(sql_conn_string)
    sql_cursor = sql_conn.cursor()
    print(f"[SUCCESS]Conection to SQL Server completed!")
    

except Exception as e:
    print(f"SQL Server connection failed:{e}")
    print(""" How to Toubleshoot:
         >1. Check server name in .env is correct
         >2. Verify SQL Server is runing
         >3. Check Windows authentication is enable
          ... 
""")

[SUCCESS]Conection to SQL Server completed!


## Connect to PostgreSQL

In [12]:
print("connect to SQL PostgreSQL...")
print(f"    Server:{pg_host}")
print(f"    User:{pg_db}")


connect to SQL PostgreSQL...
    Server:localhost
    User:dvdrental


In [13]:
try:
    pg_conn = psycopg2.connect(
        host=pg_host,
        port=pg_port,
        database=pg_db,
        user=pg_user,
        password=pg_password
    )

    pg_cursor=pg_conn.cursor()
    pg_cursor.execute("SELECT version();")
    pg_version = pg_cursor.fetchone()[0]
    print("Conected to PostgreSQL")
    print(f"    Version: {pg_version[:50]}...\n")

except psycopg2.OperationalError as e:
    print(f"Postgres Connection failed{e}")
    print(""" How to trobleshoot:
            > 1. Check Postgres is running 
            > 2. Verify user name + password
            > 3. Check db exists 
          
          ...
""")
    
except Exception as e:
    print(f"Unexpected error: {e}")
    raise

Conected to PostgreSQL
    Version: PostgreSQL 18.3 on x86_64-windows, compiled by msv...



## 4. Define the tables to migrate

### Migration Orden
- Sales.Customer (Dependencies)
- Production.ProductCategory (No Dependencies)
- Purchasing.ProductVendor (Depedencies)
- Person.Adress (No Dependencies)

In [14]:
tables_to_migrate= ['Sales.Customer', 'Production.ProductCategory', 'Purchasing.ProductVendor', 'Person.Address']
print(tables_to_migrate)

['Sales.Customer', 'Production.ProductCategory', 'Purchasing.ProductVendor', 'Person.Address']


In [15]:
print("Tables to migrate:")
for i, table in enumerate(tables_to_migrate, 1):
    print(f"    {i}.    {table}")

total_no_tbls = len(tables_to_migrate)
print(f"\nTotal number of tables to migrate: {total_no_tbls}")
    

Tables to migrate:
    1.    Sales.Customer
    2.    Production.ProductCategory
    3.    Purchasing.ProductVendor
    4.    Person.Address

Total number of tables to migrate: 4


## 5. Run pre migration checks

In [16]:
print("==" * 50)
print(">> Rows counts")
print("==" * 50)

>> Rows counts


In [17]:
test_query = "SELECT COUNT(*) AS total_rows FROM Production.ProductCategory;"
sql_cursor.execute(test_query)
count = sql_cursor.fetchone()[0]

print(f"Result: {count}")

Result: 4


In [18]:
baseline_counts = {}

try:
    for table in tables_to_migrate:
        
        row_count_query = f"SELECT COUNT(*) as total_rows FROM {table}"
        sql_cursor.execute(row_count_query)
        count = sql_cursor.fetchone()[0]

        baseline_counts[table] = count
        print(f"{table:25} {count:>20} rows")

    total_rows = sum(baseline_counts.values())
    print(f"{'-' *30}")
    print(f"{'TOTAL':25} {total_rows:>20,} rows ")     
    print(f"Baseline captured!")        
           
        
except Exception as e:
    print (f"Failed to get baseline counts: {e}")
    raise

Sales.Customer                           19820 rows
Production.ProductCategory                    4 rows
Purchasing.ProductVendor                   460 rows
Person.Address                           19614 rows
------------------------------
TOTAL                                   39,898 rows 
Baseline captured!


In [19]:
print("==" * 50)
print(">> Check 2: NULL COUNTS (Purchasing.ProductVendor )")
print("==" * 50)

>> Check 2: NULL COUNTS (Purchasing.ProductVendor )


In [20]:
quality_issues= []

try:
    print(">> Check 2: NULL COUNTS (Purchasing.ProductVendor)\n")
    sql_cursor.execute("""SELECT COUNT(*) AS null_count 
                       FROM Purchasing.ProductVendor WHERE OnOrderQty IS NULL""")
    null_OnQty= sql_cursor.fetchone()[0]
    if null_OnQty >0:
        quality_issues.append(f"  {null_OnQty} customer with NULL On Order Qty ")
    #print(quality_issues)

    print(">> Check 3: MIN ORDER QUANTITY ...")
    sql_cursor.execute("""SELECT COUNT(*) as Minimal_quantity_product  
                          FROM Purchasing.ProductVendor 
                          WHERE MinOrderQty <=1 """)
    
    Minimal_quantity_product = sql_cursor.fetchone()[0]
    if Minimal_quantity_product > 0:
        quality_issues.append(f"   - {Minimal_quantity_product:,} with only 1 product ...")
    print(quality_issues)

except Exception as e:
    print (f"Failed to get baseline counts: {e}")
    raise

print("\nCHECK 4:INVALID ADDRESS WITH OUT NUMBER")
sql_cursor.execute(""" SELECT COUNT(*) AS INVALID_ADRESS 
                        FROM Person.Address
                        WHERE AddressLine1 IS NOT NULL""")



>> Check 2: NULL COUNTS (Purchasing.ProductVendor)

>> Check 3: MIN ORDER QUANTITY ...
['  305 customer with NULL On Order Qty ', '   - 285 with only 1 product ...']

CHECK 4:INVALID ADDRESS WITH OUT NUMBER


In [44]:
conn =  pyodbc.connect(
        f"USER={sql_user};"
        f"PASSWORD={sql_password};"
        f"SERVER={sql_host};"
        f"DATABASE={sql_db};"
        f"DRIVER={{SQL Server}};"
        f"Trusted_Conection=yes;"
    )

query= 'SELECT AddressLine1 FROM Person.Address'
df = pd.read_sql_query(query,conn)
str_sql= df
df_address=pd.DataFrame(str_sql)
df_address

C:\Users\IT\AppData\Local\Temp\ipykernel_26104\2790461171.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query,conn)


,AddressLine1
0,#500-75 O'Connor Street
1,#9900 2700 Production Way
2,"00, rue Saint-Lazare"
3,"02, place de Fontenoy"
4,"035, boulevard du Montparnasse"
...,...
19609,Zur Lindung 7
19610,Zur Lindung 7
19611,Zur Lindung 764
19612,Zur Lindung 78


In [ ]:
data_list = []
def counter(df_address):
            c = 0          
            for i in df_address['AddressLine1']:
                eval = any(char.isdigit() for char in i)
                if eval is False:
                    #print("Detecting Invalid adress...")
                    data_list.append({'Wrong_AddressLine1':i})                   
                    c += 1
            return c
                    
cad = df_address
print(f"Number of registers without number in address: {counter(cad)}")
pd_invalid=pd.DataFrame(data_list)
pd_invalid

Number of registers without number in address: 207


,Wronng_AddressLine1
0,Adirondack Factory Outlet
1,Ames Plaza
2,Amity Plaza
3,Arcadia Crossing
4,Attaché de Presse
...,...
202,Wharfdale Road
203,White Mountain Mall
204,Woodbury Commons
205,Wrentham Village
